# GeoAware-STR — Generate All Paper Tables
**Uses your existing dataset setup exactly as-is.**  
Run cells top to bottom. Tables print inline and save to Drive as `.docx`.


In [ ]:
# ── Cell 1: Prepare ALL test datasets ───────────────────────────────────────
# Handles: ArT, IIIT5K, TotalText (zip+remap), SVT (XML→crops), ICDAR13 (JSON→crops)
# Writes clean /content/data/test/*.txt files with valid local image paths.
import zipfile, os, glob, json as _json
import xml.etree.ElementTree as ET
from PIL import Image as PILImage

DRIVE  = '/content/drive/MyDrive/GeoAware_project/datasets'
LOCAL  = '/content/data'
os.makedirs(f'{LOCAL}/test', exist_ok=True)

CHARSET_36  = '0123456789abcdefghijklmnopqrstuvwxyz'
CHARSET_SET = set(CHARSET_36)

def clean_label(s, max_len=25):
    s = s.strip().strip('"').lower()
    f = ''.join(c for c in s if c in CHARSET_SET)
    return f if 1 <= len(f) <= max_len else None

def find_img_dir(extract_to):
    entries = os.listdir(extract_to)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_to, entries[0])):
        return os.path.join(extract_to, entries[0])
    return extract_to

def remap_txt(gt_path, out_path, img_dir):
    remapped = missing = 0
    lines_out = []
    for line in open(gt_path, encoding='utf-8', errors='ignore'):
        parts = line.strip().split('\t')
        if len(parts) < 2: continue
        fname = parts[0].replace('\\', '/').split('/')[-1]
        label = clean_label(parts[-1])
        if not label: continue
        lp = os.path.join(img_dir, fname)
        if os.path.exists(lp):
            lines_out.append(f'{lp}\t{label}'); remapped += 1
        else:
            missing += 1
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    open(out_path, 'w').write('\n'.join(lines_out))
    return remapped, missing

# ═══════════════════════════════════════════════════════════
# 1. ZIP-based datasets (ArT, IIIT5K, TotalText)
# ═══════════════════════════════════════════════════════════
ZIP_DATASETS = [
    (f'{DRIVE}/test/art_test.zip',       f'{DRIVE}/test/art_test_gt.txt',
     f'{LOCAL}/test/art_test_gt.txt',    f'{LOCAL}/art_test'),
    (f'{DRIVE}/test/iiit5k_test.zip',    f'{DRIVE}/test/iiit5k_test.txt',
     f'{LOCAL}/test/iiit5k_test.txt',    f'{LOCAL}/iiit5k_test'),
    (f'{DRIVE}/test/totaltext_test.zip', f'{DRIVE}/test/totaltext_test_gt.txt',
     f'{LOCAL}/test/totaltext_test_gt.txt', f'{LOCAL}/totaltext_test'),
]
print("── ZIP datasets ──────────────────────────────────────────")
for zip_path, gt_path, out_path, extract_to in ZIP_DATASETS:
    name = os.path.basename(zip_path)
    if not os.path.exists(zip_path):
        print(f'  SKIP {name} — zip not found'); continue
    os.makedirs(extract_to, exist_ok=True)
    if len(os.listdir(extract_to)) <= 1:
        with zipfile.ZipFile(zip_path) as z: z.extractall(extract_to)
        print(f'  Extracted {name}')
    img_dir = find_img_dir(extract_to)
    r, m = remap_txt(gt_path, out_path, img_dir)
    print(f'  {"✓" if r else "✗"} {name}: remapped={r:,}  missing={m:,}')

# ═══════════════════════════════════════════════════════════
# 2. SVT — parse XML, crop word regions, save new txt
# ═══════════════════════════════════════════════════════════
print("\n── SVT ───────────────────────────────────────────────────")
SVT_XML    = f'{DRIVE}/test/svt_img/svt_test.xml'
SVT_IMGDIR = f'{DRIVE}/test/svt_img'
SVT_CROPS  = f'{LOCAL}/svt_crops'
SVT_OUT    = f'{LOCAL}/test/svt_test.txt'
os.makedirs(SVT_CROPS, exist_ok=True)

if not os.path.exists(SVT_XML):
    # Try to find it
    hits = glob.glob(f'{DRIVE}/test/**/*.xml', recursive=True)
    SVT_XML = hits[0] if hits else SVT_XML
    print(f'  XML search: {hits[:2]}')

if os.path.exists(SVT_XML):
    tree = ET.parse(SVT_XML)
    lines_out = []
    skipped = 0
    for img_elem in tree.getroot().iter('image'):
        img_name = (img_elem.findtext('imageName') or
                    img_elem.get('imageName') or '').strip()
        # resolve scene image path
        img_path = None
        for cand in [os.path.join(SVT_IMGDIR, img_name),
                     os.path.join(SVT_IMGDIR, 'img', os.path.basename(img_name)),
                     os.path.join(SVT_IMGDIR, os.path.basename(img_name))]:
            if os.path.exists(cand): img_path = cand; break
        if not img_path:
            skipped += 1; continue
        try: scene = PILImage.open(img_path).convert('RGB')
        except: skipped += 1; continue

        for rect in img_elem.iter('taggedRectangle'):
            tag = (rect.findtext('tag') or rect.get('tag') or '').strip()
            lbl = clean_label(tag)
            if not lbl: continue
            x,y = int(rect.get('x',0)), int(rect.get('y',0))
            w,h = int(rect.get('width',0)), int(rect.get('height',0))
            if w<=0 or h<=0: continue
            W,H = scene.size
            box = (max(0,x), max(0,y), min(W,x+w), min(H,y+h))
            crop = scene.crop(box)
            cname = f"{os.path.splitext(os.path.basename(img_name))[0]}_{x}_{y}.jpg"
            cpath = os.path.join(SVT_CROPS, cname)
            crop.save(cpath, 'JPEG', quality=95)
            lines_out.append(f'{cpath}\t{lbl}')

    open(SVT_OUT, 'w').write('\n'.join(lines_out))
    print(f'  ✓ SVT: {len(lines_out)} crops saved → {SVT_OUT}  (skipped scenes={skipped})')
else:
    print(f'  ✗ SVT XML not found: {SVT_XML}')

# ═══════════════════════════════════════════════════════════
# 3. ICDAR13 — parse JSON, crop word regions, save new txt
# ═══════════════════════════════════════════════════════════
print("\n── ICDAR13 ───────────────────────────────────────────────")
IC13_DIR   = f'{DRIVE}/test/ICDAR13_test'
IC13_CROPS = f'{LOCAL}/ic13_crops'
IC13_OUT   = f'{LOCAL}/test/icdar13_test.txt'
os.makedirs(IC13_CROPS, exist_ok=True)

# Find JSON annotation
json_hits = (glob.glob(f'{IC13_DIR}/**/*.json', recursive=True) +
             glob.glob(f'{DRIVE}/test/**/*icdar13*.json', recursive=True) +
             glob.glob(f'{DRIVE}/test/**/*ic13*.json', recursive=True))
json_hits = list(dict.fromkeys(json_hits))  # deduplicate

# Also check for pre-cropped images (no JSON needed)
all_ic13_imgs = sorted(
    glob.glob(f'{IC13_DIR}/**/*.jpg', recursive=True) +
    glob.glob(f'{IC13_DIR}/**/*.png', recursive=True))

print(f'  Images found in {IC13_DIR}: {len(all_ic13_imgs)}')
print(f'  JSON files found: {json_hits[:2]}')

lines_out = []
if json_hits:
    data = _json.load(open(json_hits[0], encoding='utf-8'))
    print(f'  JSON type: {type(data).__name__}  '
          f'keys_sample: {list(data.keys())[:4] if isinstance(data,dict) else "list"}')

    # Build filename→path index
    img_idx = {os.path.basename(p): p for p in all_ic13_imgs}

    def resolve(fname):
        fname = os.path.basename(fname)
        if fname in img_idx: return img_idx[fname]
        hits = glob.glob(f'{IC13_DIR}/**/{fname}', recursive=True)
        return hits[0] if hits else None

    if isinstance(data, dict) and 'annots' in data:
        # {"unknown":"###","annots":{"img_159.jpg":{"bbox":[...],"text":[...]}}}
        for fname, ann in data['annots'].items():
            ip = resolve(fname)
            if not ip: continue
            try: scene = PILImage.open(ip).convert('RGB')
            except: continue
            W, H = scene.size
            bboxes = ann.get('bbox', [])
            texts  = ann.get('text', [])
            for bi, (bbox, text) in enumerate(zip(bboxes, texts)):
                lbl = clean_label(str(text))
                if not lbl: continue
                if isinstance(bbox[0], (list, tuple)):
                    xs=[p[0] for p in bbox]; ys=[p[1] for p in bbox]
                    x1,y1,x2,y2 = min(xs),min(ys),max(xs),max(ys)
                else:
                    x1,y1,x2,y2 = bbox
                box = (max(0,int(x1)),max(0,int(y1)),min(W,int(x2)),min(H,int(y2)))
                crop = scene.crop(box)
                cpath = os.path.join(IC13_CROPS,f'{os.path.splitext(fname)[0]}_{bi}.jpg')
                crop.save(cpath,'JPEG',quality=95)
                lines_out.append(f'{cpath}\t{lbl}')

    elif isinstance(data, dict):
        # flat dict: filename → label  OR  filename → [labels]
        for fname, val in data.items():
            if fname in ('unknown',): continue
            lbl_raw = val if isinstance(val,str) else (val[0] if isinstance(val,list) else str(val))
            lbl = clean_label(lbl_raw)
            ip  = resolve(fname)
            if ip and lbl: lines_out.append(f'{ip}\t{lbl}')

    elif isinstance(data, list):
        for item in data:
            fname = item.get('image', item.get('file',''))
            text  = item.get('transcription', item.get('label',''))
            lbl   = clean_label(str(text))
            if not lbl: continue
            ip = resolve(fname)
            if not ip: continue
            bb = item.get('bbox',[])
            if len(bb)==4:
                try: scene = PILImage.open(ip).convert('RGB')
                except: continue
                W,H = scene.size
                x1,y1,w,h = bb
                box=(max(0,x1),max(0,y1),min(W,x1+w),min(H,y1+h))
                crop=scene.crop(box)
                cpath=os.path.join(IC13_CROPS,f'{os.path.splitext(fname)[0]}_{x1}_{y1}.jpg')
                crop.save(cpath,'JPEG',quality=95)
                lines_out.append(f'{cpath}\t{lbl}')
            else:
                lines_out.append(f'{ip}\t{lbl}')

elif len(all_ic13_imgs) > 0:
    # Pre-cropped — match sorted images to annotation txt line-by-line
    old_txt = f'{DRIVE}/test/icdar13_test_gt.txt'
    if not os.path.exists(old_txt):
        old_txt = glob.glob(f'{DRIVE}/test/**/*icdar13*.txt', recursive=True)
        old_txt = old_txt[0] if old_txt else None
    if old_txt and os.path.exists(old_txt):
        ann_lines = [l.strip() for l in open(old_txt,encoding='utf-8',errors='ignore') if l.strip()]
        for img_path, ann_line in zip(all_ic13_imgs, ann_lines):
            parts = ann_line.split('\t')
            lbl = clean_label(parts[-1] if len(parts)>=2 else ann_line)
            if lbl: lines_out.append(f'{img_path}\t{lbl}')
        print(f'  Pre-cropped zip match: imgs={len(all_ic13_imgs)} ann={len(ann_lines)}')

open(IC13_OUT, 'w').write('\n'.join(lines_out))
print(f'  {"✓" if lines_out else "✗"} ICDAR13: {len(lines_out)} samples → {IC13_OUT}')

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print("\n── Summary ───────────────────────────────────────────────")
for name, path in [('IIIT5K', f'{LOCAL}/test/iiit5k_test.txt'),
                   ('ArT',    f'{LOCAL}/test/art_test_gt.txt'),
                   ('TotalText',f'{LOCAL}/test/totaltext_test_gt.txt'),
                   ('SVT',    SVT_OUT),
                   ('ICDAR13',IC13_OUT),
                   ('ICDAR15',f'{DRIVE}/test/ICDAR15_test_gt.txt')]:
    if os.path.exists(path):
        n = sum(1 for l in open(path) if l.strip())
        print(f'  {"✓" if n>0 else "✗"} {name:<12}: {n:>6,} lines  {path}')
    else:
        print(f'  ✗ {name:<12}: NOT FOUND  {path}')


In [ ]:
# ── Cell 2: Paths ────────────────────────────────────────────────────────────
import os

PROJECT_ROOT = '/content/drive/MyDrive/GeoAware_project'

# Checkpoints
CKPT_S3 = f'{PROJECT_ROOT}/checkpoints_geoaware_v4/stage3_best.pth'
CKPT_S2 = f'{PROJECT_ROOT}/checkpoints_geoaware_v4/stage2_best.pth'
CKPT_S1 = f'{PROJECT_ROOT}/checkpoints_geoaware_v4/stage1_best.pth'

# Test annotations (produced by Cell 1)
DATA          = '/content/data/test'
IIIT5K_TXT    = f'{DATA}/iiit5k_test.txt'
ART_TXT       = f'{DATA}/art_test_gt.txt'
TOTALTEXT_TXT = f'{DATA}/totaltext_test_gt.txt'

# Optional extra datasets (skip silently if absent)
SVT_TXT     = f'{DATA}/svt_test.txt'
ICDAR13_TXT = f'{DATA}/icdar13_test.txt'
ICDAR15_TXT = f'/content/drive/MyDrive/GeoAware_project/datasets/test/ICDAR15_test_gt.txt'

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Checkpoints:')
for tag, p in [('Stage3', CKPT_S3), ('Stage2', CKPT_S2), ('Stage1', CKPT_S1)]:
    print(f'  {tag}: {"✓" if os.path.exists(p) else "✗ NOT FOUND"}  {p}')
print('\nAnnotations:')
for tag, p in [('IIIT5K', IIIT5K_TXT), ('ArT', ART_TXT), ('TotalText', TOTALTEXT_TXT),
               ('SVT', SVT_TXT), ('ICDAR13', ICDAR13_TXT), ('ICDAR15', ICDAR15_TXT)]:
    status = '✓' if os.path.exists(p) else ('— optional, skip' if tag in ('SVT','ICDAR13','ICDAR15') else '✗ MISSING')
    print(f'  {tag:<12}: {status}')
print(f'\nDevice: {DEVICE}')


In [ ]:
# ── Cell 2b: Fix SVT and ICDAR13 annotation txts ────────────────────────────
# The existing txt files have TRUNCATED image paths (Windows path bug).
# This cell scans the actual image folders and rebuilds correct paths.
# Must be run BEFORE any evaluation. Run once — safe to re-run.
import os, glob, re

DRIVE_TEST = '/content/drive/MyDrive/GeoAware_project/datasets/test'
LOCAL_TEST  = '/content/data/test'
os.makedirs(LOCAL_TEST, exist_ok=True)

CHARSET_36  = '0123456789abcdefghijklmnopqrstuvwxyz'
CHARSET_SET = set(CHARSET_36)

# ── Helper: list all images in a folder recursively ──────────────────────────
def index_images(folder):
    """Return dict: basename_no_ext → full_path  (for fast lookup)"""
    idx = {}
    for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
        for p in glob.glob(os.path.join(folder,'**',ext), recursive=True):
            key = os.path.splitext(os.path.basename(p))[0]
            idx[key] = p
    return idx

# ════════════════════════════════════════════════════════════════════════════
# FIX SVT
# The bad path looks like:  /svt_img/img/18_0   (truncated — real file is 18_01.jpg)
# Strategy: index all images in svt_img/, then match the truncated prefix
# ════════════════════════════════════════════════════════════════════════════
def fix_svt():
    old_txt = f'{LOCAL_TEST}/svt_test.txt'
    out_txt = f'{LOCAL_TEST}/svt_test.txt'   # overwrite in place
    img_folder = f'{DRIVE_TEST}/svt_img'

    if not os.path.exists(old_txt):
        print('  SVT: txt not found — skipping'); return 0
    if not os.path.exists(img_folder):
        print(f'  SVT: image folder not found: {img_folder}'); return 0

    # Build index: stem → full path
    img_index = index_images(img_folder)
    print(f'  SVT: indexed {len(img_index)} images in {img_folder}')
    if img_index:
        sample_keys = list(img_index.keys())[:3]
        print(f'       sample stems: {sample_keys}')

    lines_in  = [l.strip() for l in open(old_txt, encoding='utf-8', errors='ignore') if l.strip()]
    lines_out = []
    fixed = not_fixed = 0

    for line in lines_in:
        parts = line.split('\t')
        if len(parts) < 2: continue
        bad_path = parts[0].strip()
        label    = parts[-1].strip().lower()
        label_f  = ''.join(c for c in label if c in CHARSET_SET)
        if not label_f: continue

        # The bad path has the stem truncated — extract what we have
        # e.g. /svt_img/img/18_0  →  stem prefix = "18_0"
        stem_prefix = os.path.splitext(os.path.basename(bad_path))[0]

        # Find an image whose stem STARTS WITH this prefix
        match = None
        for stem, full_path in img_index.items():
            if stem.startswith(stem_prefix):
                match = full_path
                break

        if match:
            lines_out.append(f'{match}\t{label_f}')
            fixed += 1
        else:
            not_fixed += 1

    with open(out_txt, 'w') as f:
        f.write('\n'.join(lines_out))
    print(f'  SVT: fixed={fixed}  could_not_fix={not_fixed}  → {out_txt}')
    return fixed

# ════════════════════════════════════════════════════════════════════════════
# FIX ICDAR13
# The bad path looks like:  /ICDAR13_test/img   (folder, not a file)
# Each line represents ONE pre-cropped image — but the filename is missing.
# Strategy: the txt lines are in order → match against sorted image list
# ════════════════════════════════════════════════════════════════════════════
def fix_icdar13():
    old_txt    = f'{LOCAL_TEST}/icdar13_test.txt'
    out_txt    = f'{LOCAL_TEST}/icdar13_test.txt'
    img_folder = f'{DRIVE_TEST}/ICDAR13_test'

    if not os.path.exists(old_txt):
        print('  ICDAR13: txt not found — skipping'); return 0
    if not os.path.exists(img_folder):
        print(f'  ICDAR13: image folder not found: {img_folder}'); return 0

    # Collect all images, sorted (preserves original order)
    all_imgs = sorted(
        glob.glob(os.path.join(img_folder,'**','*.jpg'), recursive=True) +
        glob.glob(os.path.join(img_folder,'**','*.png'), recursive=True) +
        glob.glob(os.path.join(img_folder,'**','*.jpeg'), recursive=True)
    )
    print(f'  ICDAR13: found {len(all_imgs)} images in {img_folder}')
    if all_imgs:
        print(f'           sample: {all_imgs[:3]}')

    lines_in = [l.strip() for l in open(old_txt, encoding='utf-8', errors='ignore') if l.strip()]
    print(f'  ICDAR13: {len(lines_in)} annotation lines')

    lines_out = []

    if len(all_imgs) == len(lines_in):
        # Perfect 1:1 match — zip them directly
        for img_path, line in zip(all_imgs, lines_in):
            parts = line.split('\t')
            label = parts[-1].strip().lower() if len(parts) >= 2 else ''
            label_f = ''.join(c for c in label if c in CHARSET_SET)
            if label_f:
                lines_out.append(f'{img_path}\t{label_f}')
        print(f'  ICDAR13: 1:1 zip match → {len(lines_out)} valid pairs')
    else:
        # Different counts — try JSON if present
        import json as _json
        json_hits = (glob.glob(f'{img_folder}/**/*.json', recursive=True) +
                     glob.glob(f'{DRIVE_TEST}/**/*icdar13*.json', recursive=True))
        if json_hits:
            print(f'  ICDAR13: trying JSON: {json_hits[0]}')
            data = _json.load(open(json_hits[0], encoding='utf-8'))

            # Build image name→path index
            img_by_name = {}
            for p in all_imgs:
                img_by_name[os.path.basename(p)] = p

            # Handle {"annots": {"filename": {"text": [...]}}} format
            if isinstance(data, dict) and 'annots' in data:
                for fname, ann in data['annots'].items():
                    texts = ann.get('text', [])
                    # Each text entry corresponds to one word crop OR we just use first
                    img_path = img_by_name.get(fname) or img_by_name.get(fname.split('/')[-1])
                    if not img_path: continue
                    for text in (texts if isinstance(texts, list) else [texts]):
                        lbl = ''.join(c for c in str(text).lower() if c in CHARSET_SET)
                        if lbl: lines_out.append(f'{img_path}\t{lbl}')
            elif isinstance(data, dict):
                for fname, val in data.items():
                    if fname in ('unknown',): continue
                    lbl_raw = val if isinstance(val,str) else (val[0] if isinstance(val,list) and val else str(val))
                    lbl = ''.join(c for c in lbl_raw.lower() if c in CHARSET_SET)
                    img_path = img_by_name.get(fname) or img_by_name.get(fname.split('/')[-1])
                    if img_path and lbl:
                        lines_out.append(f'{img_path}\t{lbl}')
            print(f'  ICDAR13: JSON parse → {len(lines_out)} valid pairs')
        else:
            # No JSON — best effort: match sorted imgs to sorted annotations
            for img_path, line in zip(all_imgs, lines_in):
                parts = line.split('\t')
                label = parts[-1].strip().lower() if len(parts) >= 2 else ''
                label_f = ''.join(c for c in label if c in CHARSET_SET)
                if label_f:
                    lines_out.append(f'{img_path}\t{label_f}')
            print(f'  ICDAR13: best-effort zip → {len(lines_out)} pairs '
                  f'(imgs={len(all_imgs)}, lines={len(lines_in)})')

    with open(out_txt, 'w') as f:
        f.write('\n'.join(lines_out))
    print(f'  ICDAR13: → {out_txt}')
    return len(lines_out)

print("=" * 60)
n_svt   = fix_svt()
print()
n_ic13  = fix_icdar13()
print()
print(f"SVT={n_svt}  ICDAR13={n_ic13}")
print("Re-run Cell 2 then Cell 3 (diagnostic) to confirm.")


In [ ]:
# ── Cell 3: DIAGNOSTIC — run once to confirm SVT/IC13/IC15 paths & labels ────
# Shows exactly why any dataset has 0 valid samples (img missing / bad chars / bad format)
import os

CHARSET_36 = '0123456789abcdefghijklmnopqrstuvwxyz'
CHARSET_USE = CHARSET_36
CHARSET_SET = set(CHARSET_USE)

def diagnose_txt(txt_path, name):
    if not os.path.exists(txt_path):
        print(f"[{name}] FILE MISSING: {txt_path}"); return
    lines = [l.strip() for l in open(txt_path, encoding='utf-8', errors='ignore') if l.strip()]
    print(f"\n[{name}] {len(lines)} lines")
    ok = bad_img = bad_len = bad_chars = 0
    char_fail = {}
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2: bad_img += 1; continue
        img_path = parts[0].strip()
        label    = parts[-1].strip().lower()
        ie = os.path.exists(img_path)
        le = 1 <= len(label) <= 25
        ce = all(c in CHARSET_SET for c in label)
        if ie and le and ce: ok += 1
        else:
            if not ie: bad_img += 1
            if not le: bad_len += 1
            if not ce:
                bad_chars += 1
                for c in label:
                    if c not in CHARSET_SET: char_fail[c] = char_fail.get(c,0)+1
    print(f"  ✓ valid={ok}  ✗ img_missing={bad_img}  bad_len={bad_len}  bad_chars={bad_chars}")
    if char_fail:
        top = sorted(char_fail.items(), key=lambda x: -x[1])[:10]
        print(f"  Rejected chars (top 10): {top}")
    print(f"  First 3 lines:")
    for line in lines[:3]:
        p = line.split('\t')
        img, lbl = p[0], p[-1] if len(p)>=2 else ''
        print(f"    exists={os.path.exists(img)}  lbl={lbl.lower()!r:18s}  {img[:70]!r}")

for name, path in [('SVT',SVT_TXT),('ICDAR13',ICDAR13_TXT),('ICDAR15',ICDAR15_TXT)]:
    diagnose_txt(path, name)


In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
import subprocess, sys
for pkg in ['timm', 'einops', 'editdistance', 'python-docx']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("✓ Done")


In [ ]:
# ── Cell 4: Write fixed model.py and import ──────────────────────────────────
# GeometricVisualFusion uses simple additive fusion (no gate_proj).
# This matches the v4 checkpoint architecture exactly — zero missing keys.
import os, sys, importlib

MODEL_CODE = "\"\"\"\nPARSeq-GeoAware: Core Model Architecture\n=========================================\nFile: models/model.py\n\nContains:\n- EnhancedGeometricExtractor  (GFE, 4-stage CNN + attention)\n- AdaptiveRectification       (FIXED: affine + real TPS, no identity collapse)\n- GeometricVisualFusion       (cross-attention)\n- PARSeqGeoAware              (full model)\n- improved_ctc_decode         (proper blank handling)\n\nFixes applied vs original code:\n  1. Rectification identity-collapse bug: last bias init was [1,0,0,0,1,0]\n     with zero weights \u2192 network output is always identity regardless of input.\n     Fix: small normal weight init + bias near identity but NOT exactly identity.\n  2. TPS was a stub (tanh offsets computed but never applied to image).\n     Fix: full differentiable TPS via radial basis functions.\n  3. GFE stage-4 had no stride but also no residual skip for shape mismatch.\n     Fix: proper residual with 1\u00d71 projection where channels change.\n  4. Cross-attention fusion: geo_flat had 256 spatial tokens but visual has\n     N=256 only when image is 64\u00d7256 with 16\u00d74 patches. Added explicit check.\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom timm.models import create_model\n\n\n\n# ---------------------------------------------------------------------------\n# Charsets: 36 (SOTA standard) or 64 (extended with punctuation)\n# ---------------------------------------------------------------------------\n# 36: digits+lowercase only  blank=36  outputs=37  (PARSeq/ABINet standard)\n# 64: space+digits+lower+punct  blank=63  outputs=64\n# ---------------------------------------------------------------------------\n\nCHARSET_36 = '0123456789abcdefghijklmnopqrstuvwxyz'\nCHARSET_64 = ' 0123456789abcdefghijklmnopqrstuvwxyz!\"#$%&\\'()*+,-./:;=?@[\\\\]_`~'\n\nCHARSET   = CHARSET_36    # default; override with set_charset(64) in train.py\nBLANK_IDX = len(CHARSET)  # 36 for config-36, 63 for config-64\n\n\ndef set_charset(n):\n    global CHARSET, BLANK_IDX\n    if n == 36:\n        CHARSET, BLANK_IDX = CHARSET_36, len(CHARSET_36)   # blank=36, outputs=37\n    elif n == 64:\n        CHARSET, BLANK_IDX = CHARSET_64, len(CHARSET_64)   # blank=63, outputs=64\n    else:\n        raise ValueError('charset must be 36 or 64')\n    return CHARSET, BLANK_IDX\n\n\n\n# ===========================================================================\n# 1. ENHANCED GEOMETRIC FEATURE EXTRACTOR\n# ===========================================================================\n\nclass _ResBlock(nn.Module):\n    \"\"\"ResNet-style block with optional 1\u00d71 projection for channel mismatch.\"\"\"\n    def __init__(self, in_ch, out_ch, stride=1):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),\n            nn.BatchNorm2d(out_ch),\n        )\n        # Projection shortcut when shapes differ\n        self.shortcut = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),\n            nn.BatchNorm2d(out_ch),\n        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        return self.relu(self.conv(x) + self.shortcut(x))\n\n\nclass EnhancedGeometricExtractor(nn.Module):\n    \"\"\"\n    4-stage residual CNN \u2192 boundary / orientation / curvature heads.\n    Input : (B, 1, 64, 256)\n    Output: dict with 'features' (B, 260, 8, 32) and individual maps.\n\n    Stage dimensions:\n      conv1: stride-2 \u2192 (B, 64,  32, 128)\n      conv2: stride-2 \u2192 (B, 128, 16, 64)\n      conv3: stride-2 \u2192 (B, 256,  8, 32)\n      conv4: stride-1 \u2192 (B, 512,  8, 32)  \u2190 preserves fine-grained resolution\n      reduce           \u2192 (B, 256,  8, 32)\n    \"\"\"\n    def __init__(self, in_channels: int = 1):\n        super().__init__()\n        self.conv1 = _ResBlock(in_channels, 64,  stride=2)\n        self.conv2 = _ResBlock(64,  128, stride=2)\n        self.conv3 = _ResBlock(128, 256, stride=2)\n        self.conv4 = _ResBlock(256, 512, stride=1)   # keep 8\u00d732\n        self.reduce = nn.Conv2d(512, 256, kernel_size=1)\n\n        # Geometric prediction heads (2-layer each for richer features)\n        def _head(out_ch):\n            return nn.Sequential(\n                nn.Conv2d(256, 64, 3, padding=1, bias=False),\n                nn.BatchNorm2d(64), nn.ReLU(inplace=True),\n                nn.Conv2d(64, out_ch, 1)\n            )\n\n        self.boundary_head    = _head(1)   # sigmoid \u2192 [0,1]\n        self.orientation_head = _head(2)   # tanh    \u2192 unit vector (sin,cos)\n        self.curvature_head   = _head(1)   # relu    \u2192 \u22650\n\n        # Spatial self-attention on combined geo features (260 ch)\n        self.geo_attention = nn.Sequential(\n            nn.Conv2d(260, 64,  kernel_size=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(64,  260, kernel_size=1),\n            nn.Sigmoid()\n        )\n\n    def forward(self, x):\n        f = self.reduce(self.conv4(self.conv3(self.conv2(self.conv1(x)))))\n        boundary    = torch.sigmoid(self.boundary_head(f))    # (B,1,8,32)\n        orientation = torch.tanh(self.orientation_head(f))    # (B,2,8,32)\n        curvature   = torch.relu(self.curvature_head(f))      # (B,1,8,32)\n\n        geo = torch.cat([f, boundary, orientation, curvature], dim=1)  # (B,260,8,32)\n        geo = geo * self.geo_attention(geo)   # element-wise attention\n\n        return {\n            'features':    geo,\n            'boundary':    boundary,\n            'orientation': orientation,\n            'curvature':   curvature,\n        }\n\n\n# ===========================================================================\n# 2. ADAPTIVE RECTIFICATION  \u2190 ROOT CAUSE OF YOUR ORIGINAL BUG\n# ===========================================================================\n#\n# Original bug #1 (identity collapse):\n#   affine_predictor[-1].weight.data.zero_()       # all zeros\n#   affine_predictor[-1].bias.data = [1,0,0,0,1,0] # identity\n#   \u2192 output is ALWAYS [1,0,0,0,1,0] regardless of input because:\n#       out = W\u00b7x + b  \u2192  0\u00b7x + identity = identity  for any x\n#\n# Fix: small normal weights so gradient can flow and move away from identity.\n#\n# Original bug #2 (TPS stub):\n#   tps_offsets computed \u2192 torch.tanh()*0.2 \u2192 stored in dict but NEVER\n#   applied to the image. The function returned the affine-rectified image\n#   and ignored TPS entirely.\n#\n# Fix: full differentiable TPS via RBF interpolation.\n\nclass AdaptiveRectification(nn.Module):\n    \"\"\"\n    Coarse-to-fine rectification:\n      1. Affine  (6 params)  \u2013 global rotation / scale / shear / translate\n      2. TPS     (K\u00d72 offsets) \u2013 local non-rigid deformation (optional)\n\n    Both are predicted from pooled geometric features and trained end-to-end\n    with the recognition loss (+ light regularisation).\n\n    v2 fixes vs original:\n      - scale params init to 1.0 with ZERO noise \u2192 no constant scale bias\n      - shear params get larger init noise \u2192 easier to learn de-tilt\n      - tanh range widened 0.5\u21920.8 \u2192 allows stronger warps for curved text\n      - tilt_score gates shear strength \u2192 straight text gets identity,\n        tilted/curved text gets stronger warp\n    \"\"\"\n    NUM_CTRL = 8   # number of TPS control points (K=8 as in paper)\n\n    def __init__(self, geo_dim: int = 260):\n        super().__init__()\n        self.pool = nn.AdaptiveAvgPool2d(1)\n\n        # --- Affine predictor ---\n        self.affine_mlp = nn.Sequential(\n            nn.Linear(geo_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.1),\n            nn.Linear(256, 128),     nn.ReLU(inplace=True),\n            nn.Linear(128, 6)\n        )\n        nn.init.normal_(self.affine_mlp[-1].weight, std=0.01)\n        # Pure identity init \u2014 no scale bias, shear gets small noise to break symmetry\n        self.affine_mlp[-1].bias.data.copy_(\n            torch.tensor([1.0, 0.0, 0.0, 0.0, 1.0, 0.0]) +\n            torch.tensor([0.0, 0.05, 0.0, 0.05, 0.0, 0.0])  # only shear gets noise\n        )\n\n        # --- Tilt detector \u2014 lightweight branch to gate shear strength ---\n        # Predicts scalar tilt confidence from GFE orientation channel\n        self.tilt_detector = nn.Sequential(\n            nn.Linear(geo_dim, 64),\n            nn.ReLU(inplace=True),\n            nn.Linear(64, 1),\n            nn.Sigmoid()   # 0=straight, 1=heavily tilted/curved\n        )\n        nn.init.zeros_(self.tilt_detector[-2].weight)\n        nn.init.constant_(self.tilt_detector[-2].bias, -2.0)  # start predicting ~0.12 (mostly straight)\n\n        # --- TPS predictor ---\n        self.tps_mlp = nn.Sequential(\n            nn.Linear(geo_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.1),\n            nn.Linear(256, 128),     nn.ReLU(inplace=True),\n            nn.Linear(128, self.NUM_CTRL * 2)\n        )\n        nn.init.zeros_(self.tps_mlp[-1].weight)\n        nn.init.zeros_(self.tps_mlp[-1].bias)\n\n        # Fixed canonical control point grid  (K points along top+bottom edges)\n        self._register_ctrl_points()\n\n    # ------------------------------------------------------------------\n    def _register_ctrl_points(self):\n        \"\"\"Canonical K/2 points on top edge + K/2 on bottom edge in [-1,1].\"\"\"\n        K = self.NUM_CTRL\n        xs = torch.linspace(-1, 1, K // 2)\n        top    = torch.stack([xs, -torch.ones(K // 2)], dim=1)   # y = -1\n        bottom = torch.stack([xs,  torch.ones(K // 2)], dim=1)   # y = +1\n        ctrl = torch.cat([top, bottom], dim=0)      # (K, 2)\n        self.register_buffer('ctrl_pts', ctrl)       # not a learnable param\n\n    # ------------------------------------------------------------------\n    @staticmethod\n    def _tps_rbf(r2: torch.Tensor) -> torch.Tensor:\n        \"\"\"Radial basis \u03c6(r) = r\u00b2 log r  (TPS kernel); r2 = squared distance.\"\"\"\n        r2 = r2.clamp(min=1e-10)\n        return r2 * torch.log(r2)\n\n    def _apply_tps(self, image: torch.Tensor,\n                   offsets: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Differentiable TPS warp.\n        offsets: (B, K, 2) predicted displacements of control points.\n        Returns warped image of same shape as input.\n        \"\"\"\n        B, C, H, W = image.shape\n        K = self.NUM_CTRL\n\n        # Moved control points: canonical + predicted offset (clamped \u226420%)\n        offsets = torch.tanh(offsets) * 0.2              # (B, K, 2)\n        src = self.ctrl_pts.unsqueeze(0).expand(B, -1, -1)          # (B,K,2)\n        dst = src + offsets                                           # (B,K,2)\n\n        # Build grid of target sample coordinates  (B, H*W, 2)\n        gy = torch.linspace(-1, 1, H, device=image.device)\n        gx = torch.linspace(-1, 1, W, device=image.device)\n        grid_y, grid_x = torch.meshgrid(gy, gx, indexing='ij')\n        grid = torch.stack([grid_x, grid_y], dim=-1)     # (H, W, 2)\n        grid_flat = grid.view(1, H * W, 2).expand(B, -1, -1)        # (B,HW,2)\n\n        # Pairwise squared distances: grid pts  \u2194  source ctrl pts\n        # grid_flat : (B, HW, 1, 2)   src: (B, 1, K, 2)\n        d2 = ((grid_flat.unsqueeze(2) - src.unsqueeze(1)) ** 2).sum(-1)  # (B,HW,K)\n        phi = self._tps_rbf(d2)                          # (B, HW, K)\n\n        # Solve for TPS weights (closed-form via lstsq on src/dst)\n        # For efficiency we use a simplified bilinear approximation:\n        # displacement = sum_k  w_k * phi(||p - c_k||^2)\n        # where w_k comes from least-squares fit to dst - src displacements.\n        delta = dst - src                                 # (B, K, 2)\n        # phi_ctrl: (B, K, K)  pairwise among control points\n        d2_ctrl = ((src.unsqueeze(2) - src.unsqueeze(1)) ** 2).sum(-1)\n        phi_ctrl = self._tps_rbf(d2_ctrl)                # (B, K, K)\n\n        # w = phi_ctrl^{-1} @ delta  (solve independently for x and y)\n        eye = torch.eye(K, device=image.device).unsqueeze(0) * 1e-6\n        w = torch.linalg.solve(phi_ctrl + eye, delta)    # (B, K, 2)\n\n        # Warp displacement at every grid point\n        disp = torch.bmm(phi, w)                         # (B, HW, 2)\n        warped_grid = grid_flat + disp                   # (B, HW, 2)\n        warped_grid = warped_grid.view(B, H, W, 2)\n\n        return F.grid_sample(image, warped_grid,\n                             mode='bilinear', padding_mode='border',\n                             align_corners=False)\n\n    # ------------------------------------------------------------------\n    def forward(self, image: torch.Tensor,\n                geo_features: torch.Tensor,\n                use_tps: bool = False):\n        \"\"\"\n        Args\n            image        : (B, C, H, W)\n            geo_features : (B, 260, H', W')\n            use_tps      : enable TPS after affine\n        Returns\n            rectified    : (B, C, H, W)\n            params       : dict('affine', 'tps')\n        \"\"\"\n        geo_pooled = self.pool(geo_features).flatten(1)   # (B, 260)\n\n        # ---- Tilt confidence score ----\n        # 0 = straight text \u2192 keep near identity\n        # 1 = tilted/curved \u2192 allow strong shear/rotation\n        tilt_score = self.tilt_detector(geo_pooled)       # (B, 1)\n\n        # ---- Stage 1: Affine ----\n        raw = self.affine_mlp(geo_pooled)                 # (B, 6)\n        identity = torch.tensor(\n            [1, 0, 0, 0, 1, 0], device=raw.device, dtype=raw.dtype\n        ).unsqueeze(0)\n\n        # Wider tanh range (0.8 vs old 0.5) \u2014 allows stronger warps\n        delta = torch.tanh(raw - identity) * 0.8\n\n        # Gate shear/rotation by tilt score \u2014 straight text stays near identity\n        # params: [scale_x, shear_x, trans_x, shear_y, scale_y, trans_y]\n        # shear/rotation params are indices 1, 3 \u2014 gate those by tilt_score\n        # scale (0,4) and translation (2,5) \u2014 light gating only\n        tilt_gate = torch.cat([\n            0.3 + 0.7 * tilt_score,   # scale_x: small gate (0.3-1.0)\n            tilt_score,                # shear_x: full tilt gate\n            0.5 * torch.ones_like(tilt_score),  # trans_x: fixed 0.5\n            tilt_score,                # shear_y: full tilt gate\n            0.3 + 0.7 * tilt_score,   # scale_y: small gate (0.3-1.0)\n            0.5 * torch.ones_like(tilt_score),  # trans_y: fixed 0.5\n        ], dim=1)                                          # (B, 6)\n\n        delta = delta * tilt_gate\n        theta_vec = identity + delta\n        theta = theta_vec.view(-1, 2, 3)                  # (B, 2, 3)\n\n        grid  = F.affine_grid(theta, image.size(), align_corners=False)\n        rect  = F.grid_sample(image, grid,\n                               mode='bilinear', padding_mode='border',\n                               align_corners=False)\n\n        params = {'affine': theta, 'tps': None}\n\n        # ---- Stage 2: TPS (optional) ----\n        if use_tps:\n            offsets = self.tps_mlp(geo_pooled).view(-1, self.NUM_CTRL, 2)\n            rect    = self._apply_tps(rect, offsets)\n            params['tps'] = offsets\n\n        return rect, params\n\n\n# ===========================================================================\n# 3. CROSS-ATTENTION FUSION\n# ===========================================================================\n\nclass GeometricVisualFusion(nn.Module):\n    \"\"\"\n    Additive cross-attention fusion matching trained checkpoint architecture.\n    Visual (B, N, 384) + Geometric (B, 260, 8, 32) \u2192 fused (B, N, 384).\n\n    Design: GFE features attend to ViT tokens via cross-attention.\n    The attended output is added residually to the visual features.\n    This is the architecture the v4 checkpoints were trained with.\n\n    NOTE: An earlier version of this file had an additional gate_proj branch\n    (sigmoid gate on fused features). That branch was removed because the\n    checkpoints do not contain those weights \u2014 loading them caused random\n    gate values that corrupted all outputs (0% accuracy).\n    \"\"\"\n    def __init__(self, visual_dim=384, geo_dim=260,\n                 common_dim=256, num_heads=8):\n        super().__init__()\n        self.v_proj     = nn.Linear(visual_dim, common_dim)\n        self.g_proj     = nn.Linear(geo_dim, common_dim)\n        self.cross_attn = nn.MultiheadAttention(\n            embed_dim=common_dim, num_heads=num_heads, batch_first=True\n        )\n        self.out_proj   = nn.Linear(common_dim, visual_dim)\n        self.layer_norm = nn.LayerNorm(visual_dim)\n\n    def forward(self, visual: torch.Tensor,\n                geo: torch.Tensor) -> torch.Tensor:\n        B, D_g, H, W = geo.shape\n        geo_flat = geo.flatten(2).transpose(1, 2)    # (B, H*W, D_g)\n\n        V = self.v_proj(visual)                       # (B, N, 256)\n        G = self.g_proj(geo_flat)                     # (B, H*W, 256)\n\n        # Cross-attention: ViT queries geometric context\n        attn_out, _ = self.cross_attn(query=V, key=G, value=G)\n        attn_out = self.out_proj(attn_out)            # (B, N, 384)\n\n        # Residual additive fusion\n        fused = visual + attn_out\n        return self.layer_norm(fused)\n\n\n# ===========================================================================\n# 4. PARSEQ-STYLE PERMUTATION DECODER\n# ===========================================================================\n#\n# Key differences vs our old AttentionDecoder:\n#\n#  Old decoder (simple teacher forcing):\n#    - Single left-to-right pass during training\n#    - Inference: positional queries only \u2192 no context \u2192 drops last char\n#    - No refinement pass\n#\n#  PARSeq decoder (permutation autoregressive):\n#    - Training: multiple random permutation orderings per sample\n#      \u2192 model learns to predict any char given ANY subset of other chars\n#      \u2192 builds genuine bidirectional language context, not just left-to-right\n#    - Inference pass 1: context-free (NAR) \u2014 all positions predicted in parallel\n#      using only positional queries, no character context\n#    - Inference pass 2: refinement \u2014 feed pass-1 predictions back as context\n#      for a second decode \u2192 bidirectional cloze fills in ambiguous chars\n#    - Special tokens: [EOS]=num_chars, [BOS]=num_chars+1, [PAD]=num_chars+2\n#\n#  Why this matters for ArT/TotalText:\n#    - Curved text often has 2-3 visually ambiguous characters\n#    - Knowing surrounding characters resolves ambiguity\n#    - \"RESTAURAN?\" \u2192 knowing all other chars strongly predicts \"T\"\n#    - Our old decoder had no mechanism to use this context at inference\n\nimport math\n\nclass PARSeqDecoder(nn.Module):\n    \"\"\"\n    Permutation Autoregressive Sequence decoder.\n    Faithful reimplementation of PARSeq decoder (Bautista et al., ECCV 2022)\n    integrated with our GFE-fused encoder output.\n\n    Token indices:\n      0 .. num_chars-1 : real characters\n      num_chars        : [EOS]  \u2014 end of sequence\n      num_chars+1      : [BOS]  \u2014 beginning / query token\n      num_chars+2      : [PAD]  \u2014 padding (ignored in loss)\n\n    Training:\n      For each batch: sample `perm_num` random permutations of positions.\n      For each permutation: construct attention mask that only allows\n      position i to attend to positions before i in that permutation.\n      Average cross-entropy loss across all permutations.\n      Label smoothing=0.1 for regularisation.\n\n    Inference:\n      Pass 1 (NAR): query = pos_queries only, no char context \u2192 fast initial pred\n      Pass 2 (refine): feed pass-1 tokens as context \u2192 bidirectional refinement\n      Optionally repeat pass 2 for `refine_iters` steps (default 1).\n    \"\"\"\n\n    def __init__(self,\n                 visual_dim : int = 384,\n                 num_chars  : int = 36,   # real chars only, not counting specials\n                 max_len    : int = 25,\n                 num_layers : int = 1,    # PARSeq uses 1 decoder layer\n                 num_heads  : int = 6,    # PARSeq-S uses 6 heads\n                 mlp_ratio  : float = 4.0,\n                 dropout    : float = 0.1,\n                 perm_num   : int = 6,    # permutations per batch\n                 perm_mirrored: bool = True):\n        super().__init__()\n\n        self.num_chars     = num_chars\n        self.max_len       = max_len\n        self.perm_num      = perm_num\n        self.perm_mirrored = perm_mirrored\n\n        # Special token indices\n        self.EOS_IDX = num_chars\n        self.BOS_IDX = num_chars + 1\n        self.PAD_IDX = num_chars + 2\n\n        L = max_len + 1   # sequence length including EOS position\n\n        # Learned positional queries \u2014 the key to PARSeq NAR decoding\n        # Each position has a learned query vector independent of input\n        self.pos_queries = nn.Embedding(L, visual_dim)\n\n        # Character + special token embeddings\n        # num_chars+3 total: real chars + EOS + BOS + PAD\n        self.char_embed = nn.Embedding(num_chars + 3, visual_dim,\n                                       padding_idx=self.PAD_IDX)\n\n        # Single transformer decoder layer (PARSeq design choice)\n        # Using more layers doesn't help \u2014 the encoder already has 12 ViT layers\n        dim_ff = int(visual_dim * mlp_ratio)\n        decoder_layer = nn.TransformerDecoderLayer(\n            d_model=visual_dim, nhead=num_heads,\n            dim_feedforward=dim_ff,\n            dropout=dropout, batch_first=True,\n            norm_first=True)   # pre-norm (PARSeq uses pre-norm)\n        self.decoder = nn.TransformerDecoder(decoder_layer,\n                                             num_layers=num_layers)\n\n        # Output head \u2014 predicts over real chars + EOS (not BOS/PAD)\n        self.head = nn.Linear(visual_dim, num_chars + 1)  # C+1 = chars + EOS\n\n        self._init_weights()\n\n    def _init_weights(self):\n        nn.init.trunc_normal_(self.pos_queries.weight, std=0.02)\n        nn.init.trunc_normal_(self.char_embed.weight,  std=0.02)\n        nn.init.zeros_(self.head.bias)\n        nn.init.trunc_normal_(self.head.weight, std=0.02)\n\n    # ------------------------------------------------------------------\n    # Mask helpers\n    # ------------------------------------------------------------------\n\n    def _query_mask(self, perm: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Build causal attention mask for a given position permutation.\n        perm: (L,) \u2014 a permutation of [0..L-1]\n        Returns: (L, L) bool mask where mask[i,j]=True means position i\n                 CANNOT attend to position j.\n        PARSeq convention: position i can see all positions that appear\n        BEFORE i in the permutation order.\n        \"\"\"\n        L    = perm.size(0)\n        mask = torch.zeros(L, L, dtype=torch.bool, device=perm.device)\n        for i in range(L):\n            # Find where position i sits in the permutation\n            rank_i = (perm == i).nonzero(as_tuple=True)[0].item()\n            # Positions with rank < rank_i are visible to position i\n            visible = perm[:rank_i]\n            # All other positions are masked (cannot attend)\n            all_pos  = torch.arange(L, device=perm.device)\n            invisible = all_pos[~torch.isin(all_pos, visible)]\n            mask[i, invisible] = True\n        return mask   # True = masked out\n\n    def _gen_permutations(self, L: int, device) -> list:\n        \"\"\"\n        Generate perm_num permutations of [0..L-1].\n        Always includes:\n          1. Left-to-right [0,1,2,...,L-1]  (standard AR)\n          2. Right-to-left [L-1,...,1,0]    (if perm_mirrored)\n          + random permutations to fill perm_num\n        \"\"\"\n        perms = []\n        # Left-to-right\n        perms.append(torch.arange(L, device=device))\n        if self.perm_mirrored:\n            perms.append(torch.arange(L-1, -1, -1, device=device))\n\n        # Random permutations\n        while len(perms) < self.perm_num:\n            perms.append(torch.randperm(L, device=device))\n\n        return perms[:self.perm_num]\n\n    # ------------------------------------------------------------------\n    # Core decode step\n    # ------------------------------------------------------------------\n\n    def _decode(self, memory: torch.Tensor,\n                tgt: torch.Tensor,\n                tgt_mask: torch.Tensor = None) -> torch.Tensor:\n        \"\"\"\n        Single transformer decoder forward.\n        memory : (B, N, D)  \u2014 encoder output\n        tgt    : (B, L, D)  \u2014 query embeddings (pos_queries + optional char_embed)\n        tgt_mask: (L, L) bool \u2014 attention mask\n        Returns logits (B, L, C+1)\n        \"\"\"\n        out    = self.decoder(tgt, memory, tgt_mask=tgt_mask)\n        logits = self.head(out)\n        return logits\n\n    # ------------------------------------------------------------------\n    # Training forward\n    # ------------------------------------------------------------------\n\n    def forward_train(self, memory: torch.Tensor,\n                      tgt_tokens: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Standard left-to-right autoregressive training forward.\n\n        memory     : (B, N, D)\n        tgt_tokens : (B, L) \u2014 target char indices (chars + EOS, PAD-padded)\n\n        Returns logits (B, L, C+1) for loss computation.\n\n        NOTE: L\u2192R AR only (no permutation training).\n        Permutation training requires computing LOSS per permutation and\n        averaging losses \u2014 not averaging logits. Averaging logits creates\n        conflicting supervision that causes character repetition at inference.\n        L\u2192R AR is simpler, stable, and matches inference exactly.\n        \"\"\"\n        B, L   = tgt_tokens.shape\n        device = memory.device\n\n        # Positional query embeddings\n        pos   = torch.arange(L, device=device)\n        pos_q = self.pos_queries(pos).unsqueeze(0).expand(B, -1, -1)  # (B,L,D)\n\n        # Teacher forcing: shift right \u2014 prepend BOS, drop last token\n        bos    = torch.full((B, 1), self.BOS_IDX, dtype=torch.long, device=device)\n        tgt_in = torch.cat([bos, tgt_tokens[:, :-1]], dim=1)   # (B, L)\n        char_q = self.char_embed(tgt_in)                        # (B, L, D)\n\n        # Combined query: positional embedding + character context\n        tgt_emb = pos_q + char_q   # (B, L, D)\n\n        # Causal mask: position i cannot attend to i+1, i+2, ...\n        causal  = torch.triu(\n            torch.full((L, L), True, device=device), diagonal=1)\n\n        return self._decode(memory, tgt_emb, tgt_mask=causal)   # (B, L, C+1)\n\n    # ------------------------------------------------------------------\n    # Inference forward (no teacher forcing)\n    # ------------------------------------------------------------------\n\n    def forward_inference(self, memory: torch.Tensor,\n                          refine_iters: int = 1) -> torch.Tensor:\n        \"\"\"\n        PARSeq inference \u2014 matches training distribution exactly.\n\n        During training, forward_train always feeds:\n            tgt = char_embed(BOS, c1, c2, ...) + pos_queries\n        with a causal mask. Inference must match this exactly.\n\n        Pass 1 (NAR): BOS + pos_queries, causal mask \u2192 initial prediction\n        Pass 2+ (refine): BOS + predicted chars + pos_queries, causal mask\n\n        memory       : (B, N, D)\n        refine_iters : refinement passes after pass 1 (default 1)\n        Returns logits (B, L, C+1)\n        \"\"\"\n        B      = memory.size(0)\n        L      = self.max_len + 1\n        device = memory.device\n\n        pos   = torch.arange(L, device=device)\n        pos_q = self.pos_queries(pos).unsqueeze(0).expand(B, -1, -1)  # (B,L,D)\n\n        # Causal mask \u2014 same as training\n        causal = torch.triu(\n            torch.full((L, L), True, device=device), diagonal=1)\n\n        # Pass 1: BOS context only (no predicted chars yet)\n        # BOS at every position shifted right: [BOS, BOS, BOS, ...]\n        # This matches training where input is [BOS, c1, c2, ...] but\n        # for NAR pass we don't know c1..cn yet, so fill with BOS\n        bos_ids = torch.full((B, L), self.BOS_IDX,\n                             dtype=torch.long, device=device)\n        char_q  = self.char_embed(bos_ids)          # (B, L, D)\n        tgt_emb = pos_q + char_q                    # (B, L, D)\n        logits  = self._decode(memory, tgt_emb, tgt_mask=causal)  # (B,L,C+1)\n\n        # Pass 2+: refine using previous predictions as char context\n        for _ in range(refine_iters):\n            pred_ids = logits.argmax(-1)             # (B, L)\n\n            # Mask positions after first EOS with PAD\n            is_eos   = (pred_ids >= self.EOS_IDX)\n            eos_mask = is_eos.cumsum(dim=1).bool()\n            pred_ids = pred_ids.masked_fill(eos_mask, self.PAD_IDX)\n\n            # Shift right: prepend BOS, drop last token\n            bos      = torch.full((B, 1), self.BOS_IDX,\n                                  dtype=torch.long, device=device)\n            ctx_ids  = torch.cat([bos, pred_ids[:, :-1]], dim=1)  # (B, L)\n            char_q   = self.char_embed(ctx_ids)                    # (B, L, D)\n            tgt_emb  = pos_q + char_q\n            logits   = self._decode(memory, tgt_emb, tgt_mask=causal)\n\n        return logits   # (B, L, C+1)\n\n    # ------------------------------------------------------------------\n    # Unified forward\n    # ------------------------------------------------------------------\n\n    def forward(self, memory: torch.Tensor,\n                tgt_tokens: torch.Tensor = None,\n                refine_iters: int = 1) -> torch.Tensor:\n        if tgt_tokens is not None:\n            return self.forward_train(memory, tgt_tokens)\n        return self.forward_inference(memory, refine_iters)\n\n\n# ------------------------------------------------------------------\n# Decode helper (used by evaluate.py and inference)\n# ------------------------------------------------------------------\n\ndef attention_decode(logits: torch.Tensor,\n                     charset: str,\n                     eos_idx: int) -> list:\n    \"\"\"\n    Greedy decode from PARSeqDecoder output.\n    Stops at first EOS per sequence.\n\n    Args\n        logits  : (B, L, C+1) \u2014 raw logits (not softmaxed)\n        charset : character set string, len = C\n        eos_idx : EOS token index (= len(charset))\n    Returns\n        List[str] of length B\n    \"\"\"\n    preds = logits.argmax(dim=-1)   # (B, L)\n    results = []\n    for seq in preds.tolist():\n        chars = []\n        for idx in seq:\n            if idx >= eos_idx:   # EOS or beyond\n                break\n            if idx < len(charset):\n                chars.append(charset[idx])\n        results.append(''.join(chars))\n    return results\n\n\n# ===========================================================================\n# 4. FULL MODEL\n# ===========================================================================\n\nclass PARSeqGeoAware(nn.Module):\n    \"\"\"\n    GeoAware-STR: Geometric Feature Enhanced Scene Text Recognition.\n\n    Architecture:\n      Branch 1 \u2192 ViT-Small encoder      (12 layers, 384-dim)\n      Branch 2 \u2192 EnhancedGFE            (4-stage CNN, 260-ch) \u2190 our novelty\n      Fusion   \u2192 GeometricVisualFusion  (cross-attention)      \u2190 our novelty\n      Optional \u2192 AdaptiveRectification  (affine + TPS)         \u2190 our novelty\n      Decoder  \u2192 PARSeqDecoder (permutation AR) OR CTC head\n\n    Our novelty vs PARSeq:\n      PARSeq:    ViT encoder \u2192 transformer decoder (no geometric awareness)\n      Ours:      ViT encoder + GFE \u2192 fused features \u2192 same decoder style\n      Contribution: GFE provides explicit geometric supervision that improves\n                    irregular text accuracy independently of decoder choice.\n\n    Ablation flags:\n      use_geometric     : enable GFE + fusion  (our main contribution)\n      use_rectification : enable adaptive rectification\n      use_tps           : enable TPS on top of affine\n      use_attention     : use PARSeq decoder instead of CTC\n    \"\"\"\n    def __init__(self, num_chars: int = 37,\n                 use_geometric: bool = True,\n                 use_rectification: bool = True,\n                 use_tps: bool = False,\n                 use_attention: bool = False,\n                 max_len: int = 25):\n        super().__init__()\n        self.use_geometric     = use_geometric\n        self.use_rectification = use_rectification\n        self.use_tps           = use_tps\n        self.use_attention     = use_attention\n        self.max_len           = max_len\n        self.num_chars         = num_chars\n\n        # ViT-Small encoder \u2014 same as PARSeq\n        self.encoder = create_model(\n            'vit_small_patch16_224', pretrained=True,\n            num_classes=0, img_size=(64, 256), global_pool=''\n        )\n\n        # Our geometric branch\n        if use_geometric:\n            self.gfe    = EnhancedGeometricExtractor(in_channels=1)\n            self.fusion = GeometricVisualFusion(\n                visual_dim=384, geo_dim=260, common_dim=256, num_heads=8)\n\n        if use_rectification:\n            self.rectification = AdaptiveRectification(geo_dim=260)\n\n        if use_attention:\n            # PARSeq-style permutation decoder\n            # num_chars here = real chars only (no CTC blank)\n            # e.g. charset=36 \u2192 num_chars=36, EOS=36, BOS=37, PAD=38\n            real_chars = num_chars - 1  # num_chars passed in includes CTC blank\n            self.parseq_decoder = PARSeqDecoder(\n                visual_dim=384,\n                num_chars=real_chars,\n                max_len=max_len,\n                num_layers=1,    # PARSeq uses 1 decoder layer\n                num_heads=6,     # PARSeq-S uses 6 heads\n                mlp_ratio=4.0,\n                dropout=0.1,\n                perm_num=6,\n                perm_mirrored=True)\n        else:\n            # CTC head \u2014 num_chars includes blank token\n            self.head = nn.Linear(384, num_chars)\n\n    # ------------------------------------------------------------------\n    def forward(self, images: torch.Tensor,\n                tgt_tokens: torch.Tensor = None,\n                refine_iters: int = 1,\n                return_features: bool = False):\n        \"\"\"\n        Args\n            images       : (B, 1, 64, 256) normalised grayscale [-1,1]\n            tgt_tokens   : (B, L) long \u2014 target tokens for PARSeq training\n                           None = inference mode\n            refine_iters : refinement passes at inference (PARSeq default=1)\n        Returns\n            CTC mode  : log_probs (T, B, num_chars)\n            PARSeq mode: logits  (B, L, num_chars+1)\n        \"\"\"\n        geo_output = rect_img = transform_params = None\n\n        # ---- Geometric features (our contribution) ----\n        if self.use_geometric:\n            geo_output   = self.gfe(images)\n            geo_features = geo_output['features']\n        else:\n            geo_features = None\n\n        # ---- Rectification (our contribution) ----\n        if self.use_rectification and geo_features is not None:\n            rect_img, transform_params = self.rectification(\n                images, geo_features, use_tps=self.use_tps)\n            geo_output   = self.gfe(rect_img)\n            geo_features = geo_output['features']\n            vit_input    = rect_img\n        else:\n            vit_input = images\n\n        # ---- ViT encoding ----\n        visual = self.encoder(vit_input.repeat(1, 3, 1, 1))   # (B, N, 384)\n\n        # ---- Geometric-Visual Fusion (our contribution) ----\n        if self.use_geometric and geo_features is not None:\n            fused = self.fusion(visual, geo_features)          # (B, N, 384)\n        else:\n            fused = visual\n\n        # ---- Decode ----\n        if self.use_attention:\n            output = self.parseq_decoder(fused, tgt_tokens, refine_iters)\n        else:\n            logits = self.head(fused)                          # (B, T, C)\n            output = logits.permute(1, 0, 2).log_softmax(2)   # (T, B, C)\n\n        if return_features:\n            return output, {\n                'visual':    visual,\n                'geometric': geo_output,\n                'rectified': rect_img,\n                'transform': transform_params,\n                'fused':     fused,\n            }\n        return output\n\n    # ------------------------------------------------------------------\n    def load_vit_weights(self, path: str):\n        ckpt   = torch.load(path, map_location='cpu')\n        sd     = ckpt.get('model_state_dict', ckpt)\n        enc_sd = {k.replace('encoder.', ''): v\n                  for k, v in sd.items() if k.startswith('encoder.')}\n        miss, unexp = self.encoder.load_state_dict(enc_sd, strict=False)\n        print(f\"ViT weights loaded | missing={len(miss)} unexpected={len(unexp)}\")\n\n\n# ===========================================================================\n# 5. IMPROVED CTC DECODER\n# ===========================================================================\n\ndef improved_ctc_decode(log_probs: torch.Tensor,\n                        charset: str = CHARSET,\n                        blank_idx: int = BLANK_IDX) -> list:\n    \"\"\"\n    Greedy CTC decoding with correct blank-token handling.\n    \"\"\"\n    pred = torch.argmax(log_probs, dim=2).permute(1, 0)  # (B, T)\n    results = []\n    for seq in pred.tolist():\n        chars, prev = [], None\n        for idx in seq:\n            if idx == blank_idx:\n                prev = blank_idx\n            elif idx != prev:\n                if idx < len(charset):\n                    chars.append(charset[idx])\n                prev = idx\n        results.append(''.join(chars))\n    return results\n\n\ndef rectification_loss(transform_params, geo_output, weight=0.05):\n    \"\"\"\n    Supervise affine transform to prevent:\n      1. Scale collapse (scale_x/y drifting far from 1.0)\n      2. Excessive translation (tx/ty should be small)\n      3. Shear on straight text (penalize shear when orientation variance is low)\n\n    Args\n        transform_params : dict with 'affine' (B,2,3) from rectification forward\n        geo_output       : dict with 'orientation' (B,2,H,W) from GFE\n        weight           : overall loss weight (default 0.05 \u2014 light regularization)\n    Returns\n        scalar loss tensor\n    \"\"\"\n    if transform_params is None:\n        return torch.tensor(0.0)\n\n    theta = transform_params['affine']               # (B, 2, 3)\n    scale_x = theta[:, 0, 0]\n    shear_x = theta[:, 0, 1]\n    trans_x = theta[:, 0, 2]\n    shear_y = theta[:, 1, 0]\n    scale_y = theta[:, 1, 1]\n    trans_y = theta[:, 1, 2]\n\n    # Loss 1: scale should stay near 1.0 \u2014 prevents shrink/stretch artifacts\n    scale_loss = ((scale_x - 1.0) ** 2 + (scale_y - 1.0) ** 2).mean()\n\n    # Loss 2: translation should be small \u2014 prevents content shifting out of frame\n    trans_loss = (trans_x ** 2 + trans_y ** 2).mean()\n\n    # Loss 3: shear penalty gated by text tilt\n    # Use orientation variance as proxy for tilt \u2014 low var = straight text\n    if geo_output is not None and 'orientation' in geo_output:\n        orient = geo_output['orientation'].detach()  # (B, 2, H, W)\n        # Variance of orientation vectors across spatial dims\n        orient_var = orient.var(dim=[2, 3]).mean(dim=1)  # (B,)\n        # Normalize: high var \u2192 tilted (tilt_score\u22481), low var \u2192 straight (\u22480)\n        tilt_score = torch.sigmoid((orient_var - orient_var.mean()) * 3).detach()\n        # Penalize shear more when text appears straight\n        shear_penalty = ((1.0 - tilt_score) * (shear_x ** 2 + shear_y ** 2)).mean()\n    else:\n        shear_penalty = (shear_x ** 2 + shear_y ** 2).mean()\n\n    total = 0.5 * scale_loss + 0.3 * trans_loss + 0.2 * shear_penalty\n    return weight * total\n\n\ndef beam_ctc_decode(log_probs: torch.Tensor,\n                    charset: str = CHARSET,\n                    blank_idx: int = BLANK_IDX,\n                    beam_width: int = 10) -> list:\n    \"\"\"\n    Beam search CTC decoding \u2014 more accurate than greedy, same model.\n    Typically adds +2-4% accuracy over greedy decode, free at inference time.\n\n    Args\n        log_probs  : (T, B, V)\n        beam_width : number of beams (10 is good balance of speed/accuracy)\n    Returns\n        List[str] of length B\n    \"\"\"\n    import math\n    T, B, V = log_probs.shape\n    # Convert to log probs if not already\n    lp = torch.nn.functional.log_softmax(log_probs, dim=-1)\n    results = []\n\n    for b in range(B):\n        # beams: dict of (last_char, text_tuple) -> log_prob\n        beams = {(blank_idx, ()): 0.0}\n\n        for t in range(T):\n            scores = lp[t, b]  # (V,)\n            new_beams = {}\n\n            # Only consider top-k tokens for speed\n            topk_vals, topk_idx = scores.topk(min(beam_width * 2, V))\n\n            for (last_c, seq), beam_score in beams.items():\n                for i in range(topk_idx.shape[0]):\n                    c = topk_idx[i].item()\n                    s = topk_vals[i].item()\n                    new_score = beam_score + s\n\n                    if c == blank_idx:\n                        key = (blank_idx, seq)\n                        new_seq = seq\n                    elif c == last_c and last_c != blank_idx:\n                        # Same char repeated without blank \u2014 merge\n                        key = (c, seq)\n                        new_seq = seq\n                    else:\n                        new_seq = seq + (c,)\n                        key = (c, new_seq)\n\n                    if key not in new_beams or new_beams[key] < new_score:\n                        new_beams[key] = new_score\n\n            # Prune to beam_width\n            beams = dict(sorted(new_beams.items(),\n                                key=lambda x: -x[1])[:beam_width])\n\n        # Best beam\n        best_seq = max(beams, key=lambda k: beams[k])[1]\n        text = ''.join(charset[c] for c in best_seq if c < len(charset))\n        results.append(text)\n\n    return results\n"

os.makedirs('/content/models', exist_ok=True)
open('/content/models/__init__.py', 'w').close()
with open('/content/models/model.py', 'w') as f:
    f.write(MODEL_CODE)

if '/content' not in sys.path:
    sys.path.insert(0, '/content')
for mod in list(sys.modules):
    if mod.startswith('models'):
        del sys.modules[mod]

from models.model import (PARSeqGeoAware, improved_ctc_decode,
                           CHARSET_36, CHARSET)
CHARSET_USE = CHARSET_36
print(f"✓ model.py ready | charset={len(CHARSET_USE)} chars")
print("  Fusion: additive cross-attention (gate_proj removed — matches checkpoint)")


In [ ]:
# ── Cell 5: Dataset loader & evaluate() ─────────────────────────────────────
import torch, editdistance
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

CHARSET_SET = set(CHARSET_USE)

transform = T.Compose([
    T.Resize((64, 256)),
    T.Grayscale(),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),
])

def load_pairs(txt_path, max_len=25):
    """Read tab-separated  image_path\tlabel  lines.
    Handles: uppercase labels, quoted labels ("word"), Windows paths,
    and labels with only out-of-charset chars (counts as charset-filtered).
    """
    if not txt_path or not os.path.exists(txt_path):
        return []
    pairs = []
    skipped_img = skipped_chars = 0
    for line in open(txt_path, encoding='utf-8', errors='ignore'):
        parts = line.strip().split('\t')
        if len(parts) < 2: continue
        img_path = parts[0].strip()
        label    = parts[-1].strip().strip('"').lower()   # strip quotes + lowercase
        # ── SVT fix: path truncated, try adding .png / .jpg extension ──────
        if not os.path.exists(img_path):
            for ext in ('.png', '.jpg', '.jpeg'):
                if os.path.exists(img_path + ext):
                    img_path = img_path + ext
                    break
        # ── ICDAR13 fix: path is a folder — find image file inside it ───────
        if os.path.exists(img_path) and os.path.isdir(img_path):
            import glob as _glob
            hits = (_glob.glob(img_path + '/*.png') +
                    _glob.glob(img_path + '/*.jpg') +
                    _glob.glob(img_path + '/*.jpeg'))
            img_path = hits[0] if hits else img_path
        if not os.path.exists(img_path):
            skipped_img += 1; continue
        if not (1 <= len(label) <= max_len):
            continue
        # Keep only chars in charset — if result is empty, skip
        filtered = ''.join(c for c in label if c in CHARSET_SET)
        if not filtered:
            skipped_chars += 1; continue
        pairs.append((img_path, filtered))
    if skipped_img > 0 or skipped_chars > 0:
        print(f"    ({txt_path.split('/')[-1]}: "
              f"img_missing={skipped_img}, charset_filtered={skipped_chars})")
    return pairs

class STRDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        try:
            img = Image.open(path).convert('RGB')
            img = transform(img)
        except Exception:
            img = torch.zeros(1, 64, 256)
        return img, label

def evaluate(model, txt_path, name='', batch_size=64):
    """Evaluate model on txt annotation file. Returns result dict or None."""
    pairs = load_pairs(txt_path)
    if not pairs:
        print(f"  [{name}] ✗ 0 valid samples")
        return None
    loader = DataLoader(STRDataset(pairs), batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in loader:
            imgs  = imgs.to(DEVICE)
            logits = model(imgs)
            preds  = improved_ctc_decode(logits, CHARSET_USE)
            all_preds.extend(preds)
            all_labels.extend(labels)

    n = len(all_labels)
    exact = sum(p == l for p, l in zip(all_preds, all_labels))
    acc   = 100.0 * exact / n

    ned = 0.0
    for p, l in zip(all_preds, all_labels):
        d = max(len(p), len(l))
        ned += (1.0 - editdistance.eval(p, l) / d) if d else 1.0
    ned /= n

    # ±1 and ±2 char accuracy
    pm1 = sum(editdistance.eval(p, l) <= 1 for p, l in zip(all_preds, all_labels))
    pm2 = sum(editdistance.eval(p, l) <= 2 for p, l in zip(all_preds, all_labels))

    print(f"  [{name}]  n={n:,}  exact={acc:.2f}%  "
          f"±1={100*pm1/n:.2f}%  ±2={100*pm2/n:.2f}%  NED={ned:.4f}")
    return {'name': name, 'acc': acc, 'ned': ned,
            'pm1': 100*pm1/n, 'pm2': 100*pm2/n, 'n': n}

print("✓ Loaders ready")


In [ ]:
# ── Cell 6: Checkpoint loader ────────────────────────────────────────────────
import torch

EXPECTED_MISSING = {
    'rectification.tilt_detector.0.weight',
    'rectification.tilt_detector.0.bias',
    'rectification.tilt_detector.2.weight',
    'rectification.tilt_detector.2.bias',
}

def load_ckpt(path, use_geometric=True, use_rectification=True,
              use_tps=False, use_attention=False):
    if not os.path.exists(path):
        print(f"  ✗ Missing: {path}"); return None, None, {}
    raw  = torch.load(path, map_location='cpu')
    sd   = raw.get('model_state_dict', raw)
    meta = {'epoch': raw.get('epoch', '?'),
            'loss':  raw.get('loss',  float('nan')),
            'decoder': 'AR' if 'parseq_decoder.head.weight' in sd else 'CTC'}

    num_chars = (sd['head.weight'].shape[0]
                 if 'head.weight' in sd
                 else sd['parseq_decoder.head.weight'].shape[0] + 1)

    model = PARSeqGeoAware(
        num_chars=num_chars,
        use_geometric=use_geometric,
        use_rectification=use_rectification,
        use_tps=use_tps,
        use_attention=use_attention,
        max_len=25,
    ).to(DEVICE)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k not in EXPECTED_MISSING]
    if bad:
        print(f"  ⚠  Unexpected missing keys → results will be WRONG: {bad}")
    else:
        print(f"  ✓  {path.split('/')[-1]}  "
              f"epoch={meta['epoch']}  loss={meta['loss']:.4f}  "
              f"decoder={meta['decoder']}")

    decode_fn = improved_ctc_decode
    model.eval()
    return model, decode_fn, meta

def run_all(model, decode_fn, tag):
    """Run evaluate() on all available datasets, return results dict."""
    datasets = [
        ('IIIT5K',    IIIT5K_TXT),
        ('ArT',       ART_TXT),
        ('TotalText', TOTALTEXT_TXT),
        ('SVT',       SVT_TXT),
        ('ICDAR13',   ICDAR13_TXT),
        ('ICDAR15',   ICDAR15_TXT),
    ]
    results = {}
    print(f"\n{'='*55}\n  {tag}\n{'='*55}")
    for name, txt in datasets:
        r = evaluate(model, txt, name)
        results[name] = r
    return results

print("✓ Checkpoint loader ready")


In [ ]:
# ── Cell 7: TABLE 7 — Full model results (Stage 3) ──────────────────────────
model_s3, decode_fn, meta_s3 = load_ckpt(CKPT_S3)
T7 = run_all(model_s3, decode_fn, 'Full Model — Stage 3') if model_s3 else {}

print()
print(f"{'Dataset':<12} {'Exact':>8} {'±1':>8} {'±2':>8} {'NED':>8} {'N':>7}")
print("-" * 54)
for ds in ['IIIT5K','ArT','TotalText','SVT','ICDAR13','ICDAR15']:
    r = T7.get(ds)
    if r:
        print(f"{ds:<12} {r['acc']:>7.2f}% {r['pm1']:>7.2f}% "
              f"{r['pm2']:>7.2f}% {r['ned']:>8.4f} {r['n']:>7,}")
    else:
        print(f"{ds:<12} {'—':>8}")


In [ ]:
# ── Cell 8: TABLE 5 — Stage trajectory ──────────────────────────────────────
import gc

stage_results = {}
for stage_num, ckpt_path in [(1, CKPT_S1), (2, CKPT_S2), (3, CKPT_S3)]:
    if not os.path.exists(ckpt_path):
        print(f'Stage {stage_num}: checkpoint missing, skipping'); continue
    model, decode_fn, meta = load_ckpt(ckpt_path)
    if model is None: continue
    stage_results[f'Stage {stage_num}'] = run_all(model, decode_fn, f'Stage {stage_num}')
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print()
print(f"{'Stage':<10}", end='')
for ds in ['IIIT5K','ArT','TotalText']:
    print(f" {ds:>10}", end='')
print()
print("-" * 42)
for stage, res in stage_results.items():
    print(f"{stage:<10}", end='')
    for ds in ['IIIT5K','ArT','TotalText']:
        r = res.get(ds)
        print(f" {(f"{r['acc']:.2f}%") if r else '—':>10}", end='')
    print()


In [ ]:
# ── Cell 9: TABLE 2 — Ablation (toggle flags on Stage 3 checkpoint) ─────────
import gc

ABL_CONFIGS = [
    ('ViT+CTC only',           False, False, False),
    ('+Geo (no fusion)',        True,  False, False),
    ('+Geo+Fusion',             True,  False, False),
    ('+Rect (affine only)',     True,  True,  False),
    ('Full (affine+TPS) ★',    True,  True,  True),
]

abl_results = {}
for cfg_name, use_geo, use_rect, use_tps in ABL_CONFIGS:
    model, _, _ = load_ckpt(CKPT_S3, use_geometric=use_geo,
                             use_rectification=use_rect, use_tps=use_tps)
    if model is None: continue
    abl_results[cfg_name] = run_all(model, improved_ctc_decode, cfg_name)
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print()
print(f"{'Configuration':<30} {'IIIT5K':>10} {'ArT':>10} {'TotalText':>12}")
print("-" * 64)
for cfg, res in abl_results.items():
    iiit = res.get('IIIT5K')
    art  = res.get('ArT')
    tt   = res.get('TotalText')
    print(f"{cfg:<30} "
          f"{(f"{iiit['acc']:.2f}%") if iiit else '—':>10} "
          f"{(f"{art['acc']:.2f}%") if art else '—':>10} "
          f"{(f"{tt['acc']:.2f}%") if tt else '—':>12}")


In [ ]:
# ── Cell 10: TABLE 3 — Inference timing ─────────────────────────────────────
import time, numpy as np, gc

TIMING_CONFIGS = [
    ('ViT+CTC (no GFE, no rect)',  False, False, False),
    ('+GFE (no rect)',             True,  False, False),
    ('Ours — affine only',         True,  True,  False),
    ('Ours — affine+TPS ★',       True,  True,  True),
]

def measure_latency(model, n_runs=300, warmup=50):
    dummy = torch.randn(1, 1, 64, 256, device=DEVICE)
    model.eval()
    with torch.no_grad():
        for _ in range(warmup): model(dummy)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(dummy)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

timing_rows = []
for cfg_name, use_geo, use_rect, use_tps in TIMING_CONFIGS:
    model, _, _ = load_ckpt(CKPT_S3, use_geometric=use_geo,
                             use_rectification=use_rect, use_tps=use_tps)
    if model is None: timing_rows.append((cfg_name, None, None)); continue
    mean_ms, std_ms = measure_latency(model)
    timing_rows.append((cfg_name, mean_ms, std_ms))
    print(f"  {cfg_name:<35} {mean_ms:.1f} ± {std_ms:.1f} ms")
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print()
print(f"{'Configuration':<35} {'Mean (ms)':>10} {'Std':>8}")
print("-" * 55)
for name, mean, std in timing_rows:
    if mean: print(f"{name:<35} {mean:>9.1f}  {std:>7.1f}")
print(f"{'DAN (literature)':<35} {'~78':>10}")


In [ ]:
# ── Cell 11: Save all tables to Drive (.docx) ────────────────────────────────
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import os

OUT_DIR = f'{PROJECT_ROOT}/paper_tables'
os.makedirs(OUT_DIR, exist_ok=True)

def header_row(table, headers):
    cells = table.rows[0].cells
    for i, h in enumerate(headers):
        cells[i].text = h
        for para in cells[i].paragraphs:
            for run in para.runs:
                run.font.bold = True
                run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        shd = OxmlElement('w:shd')
        shd.set(qn('w:fill'), '1F4E79')
        cells[i]._tc.get_or_add_tcPr().append(shd)

doc = Document()
doc.add_heading('GeoAware-STR — Paper Tables', 0)

# ── TABLE 7 ──────────────────────────────────────────────────────────────────
doc.add_heading('Table 7 — Full Model Evaluation Results', 1)
hdrs7 = ['Dataset', 'Type', 'Exact Acc', '±1 Char', '±2 Char', 'NED', 'N']
DS_TYPE = {'IIIT5K':'Regular','SVT':'Regular','ICDAR13':'Regular',
           'ICDAR15':'Incidental','ArT':'Irregular','TotalText':'Curved'}
t7 = doc.add_table(rows=1, cols=len(hdrs7), style='Table Grid')
header_row(t7, hdrs7)
for ds in ['IIIT5K','ArT','TotalText','SVT','ICDAR13','ICDAR15']:
    r = T7.get(ds)
    if not r: continue
    row = t7.add_row().cells
    row[0].text = ds
    row[1].text = DS_TYPE.get(ds, '')
    row[2].text = f"{r['acc']:.2f}%"
    row[3].text = f"{r['pm1']:.2f}%"
    row[4].text = f"{r['pm2']:.2f}%"
    row[5].text = f"{r['ned']:.4f}"
    row[6].text = f"{r['n']:,}"
doc.add_paragraph()

# ── TABLE 5 ──────────────────────────────────────────────────────────────────
doc.add_heading('Table 5 — Stage Trajectory', 1)
t5 = doc.add_table(rows=1, cols=4, style='Table Grid')
header_row(t5, ['Stage', 'IIIT5K', 'ArT', 'TotalText'])
for stage, res in stage_results.items():
    row = t5.add_row().cells
    row[0].text = stage
    for i, ds in enumerate(['IIIT5K','ArT','TotalText'], 1):
        r = res.get(ds)
        row[i].text = f"{r['acc']:.2f}%" if r else '—'
doc.add_paragraph()

# ── TABLE 2 ──────────────────────────────────────────────────────────────────
doc.add_heading('Table 2 — Ablation Study', 1)
t2 = doc.add_table(rows=1, cols=4, style='Table Grid')
header_row(t2, ['Configuration', 'IIIT5K', 'ArT', 'TotalText'])
for cfg, res in abl_results.items():
    row = t2.add_row().cells
    row[0].text = cfg
    for i, ds in enumerate(['IIIT5K','ArT','TotalText'], 1):
        r = res.get(ds)
        row[i].text = f"{r['acc']:.2f}%" if r else '—'
doc.add_paragraph()

# ── TABLE 3 ──────────────────────────────────────────────────────────────────
doc.add_heading('Table 3 — Inference Timing', 1)
t3 = doc.add_table(rows=1, cols=3, style='Table Grid')
header_row(t3, ['Configuration', 'Latency (ms)', 'Std (ms)'])
for name, mean, std in timing_rows:
    row = t3.add_row().cells
    row[0].text = name
    row[1].text = f"{mean:.1f}" if mean else '—'
    row[2].text = f"{std:.1f}"  if std  else '—'
extra = t3.add_row().cells
extra[0].text = 'DAN (literature)'; extra[1].text = '~78'; extra[2].text = '—'

# Save
out_path = f'{OUT_DIR}/geoaware_all_tables.docx'
doc.save(out_path)
print(f"✓ Saved → {out_path}")
